# Loan repayment schedule

Edit the **Parameters** cell and run all cells, or run headless with papermill:

```
uv run papermill notebooks/generate_schedule.ipynb out.ipynb -p loan_id LN-001 -p principal 250000.00
```

The notebook holds no calculation logic: everything is in the `repayment_schedule` package.

In [ ]:
# Parameters (papermill). Field names match LoanInputs and the future Cloud Run JSON payload.
# Money as strings keeps it exact; use None for fields that do not apply.
loan_id = "LN-0001"
principal = "250000.00"        # gross balance on day one, incl. capitalised fees; None if net_advance given
net_advance = None             # only with retained_months > 0, instead of principal
annual_rate = "0.0895"
day_count = "ACT_365"          # ACT_365 | ACT_360 | 30E_360
structure = "AMORTISING"       # AMORTISING | INTEREST_ONLY
interest_treatment = "PAID"    # PAID | ROLLED (interest-only only)
retained_months = 0
disbursement_date = "2026-10-15"
first_payment_date = "2026-11-15"
term_months = 60
balloon_amount = "0.00"        # amortising only
capitalised_fees = "0.00"      # EIR only; does not change the customer schedule
holiday_calendar = "ENGLAND_WALES"

output_dir = "."               # where the CSV is written

In [ ]:
from IPython.display import display

from repayment_schedule import LoanInputs, ScheduleValidationError, generate_schedule
from repayment_schedule.export import (
    MONEY_COLUMNS, checks_dataframe, schedule_dataframe, summary_dataframe, write_csv,
)

params = {name: globals()[name] for name in LoanInputs.field_names()}
result = generate_schedule(LoanInputs(**params), raise_on_failure=False)

In [ ]:
display(summary_dataframe(result))
for warning in result.warnings:
    print("WARNING:", warning)

In [ ]:
display(checks_dataframe(result))
if not result.passed:
    raise ScheduleValidationError(result)  # stop: nothing is shown or exported

In [ ]:
display(
    schedule_dataframe(result)
    .style.format({col: "{:,.2f}" for col in MONEY_COLUMNS})
    .hide(axis="index")
)
csv_path = write_csv(result, output_dir)
print(f"Schedule written to {csv_path.resolve()}")